# 04 — Treinamento do Modelo e Avaliação

**Input:** `data/features.parquet`  
**Output:** `model.pkl`, `metrics.json`, `shap_plots/`

Pipeline:
1. Split temporal — Train: Jan–Abr / Test: Mai–Jun
2. XGBoost com `scale_pos_weight` para classes desbalanceadas
3. Avaliação: F1, precision, recall, AUC — **não accuracy**
4. SHAP values para explicabilidade
5. Avaliação para 3 horizontes: label_1h, label_2h, label_4h

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from src.utils import set_seeds
from src.model import train_model, evaluate_model, save_model, save_metrics, FEATURE_COLS

set_seeds(42)

DATA_DIR = '../data'
SHAP_DIR = Path('../shap_plots')
SHAP_DIR.mkdir(exist_ok=True)

## 1. Carregamento das Features

In [ ]:
features = pd.read_parquet(f'{DATA_DIR}/features.parquet')
print(f'Features: {features.shape}')
print(f'Período: {features["timestamp"].min()} → {features["timestamp"].max()}')
for col in ['label_1h', 'label_2h', 'label_4h']:
    print(f'{col}: {features[col].mean()*100:.4f}% positivos')

## 2. Split Temporal

In [ ]:
train_mask = features['timestamp'] <= '2025-04-30 23:59:59'
test_mask = features['timestamp'] >= '2025-05-01'
print(f'Train (Jan–Abr): {train_mask.sum():,} registros')
print(f'Test  (Mai–Jun): {test_mask.sum():,} registros')

## 3. Treinamento — 3 Horizontes

In [ ]:
all_metrics = {}
models = {}

for label_col in ['label_1h', 'label_2h', 'label_4h']:
    print(f'\n--- Treinando para {label_col} ---')
    model, metrics = train_model(features, label_col=label_col)
    models[label_col] = model
    all_metrics[label_col] = metrics
    print(f'  F1={metrics["f1"]:.3f}  Precision={metrics["precision"]:.3f}  Recall={metrics["recall"]:.3f}  AUC={metrics["auc"]:.3f}')

## 4. Avaliação por Threshold

In [ ]:
best_label = max(all_metrics, key=lambda k: all_metrics[k]['f1'])
print(f'Melhor horizonte: {best_label} (F1={all_metrics[best_label]["f1"]:.3f})')

X_test = features.loc[test_mask, FEATURE_COLS]
y_test = features.loc[test_mask, best_label]

threshold_results = evaluate_model(models[best_label], X_test, y_test)
print('\nMétricas por threshold:')
for r in threshold_results:
    print(f'  threshold={r["threshold"]}  F1={r["f1"]:.3f}  Precision={r["precision"]:.3f}  Recall={r["recall"]:.3f}')

## 5. SHAP — Importância das Features

In [ ]:
best_model = models[best_label]
X_shap = features.loc[test_mask, FEATURE_COLS].head(2000)  # amostra para velocidade

# Workaround for shap/shap#4202: xgboost>=3.1 serializes base_score as a
# bracketed array string (e.g. '[5E-1]'), which shap's UBJSON loader doesn't
# unwrap, raising ValueError. Strip the brackets after decode (value is
# unaffected — only the string formatting changed upstream).
import shap.explainers._tree as _shap_tree
_orig_decode_ubjson = _shap_tree.decode_ubjson_buffer
def _decode_ubjson_fixed(fd):
    result = _orig_decode_ubjson(fd)
    try:
        p = result['learner']['learner_model_param']
        if isinstance(p['base_score'], str):
            p['base_score'] = p['base_score'].strip('[]')
    except (KeyError, TypeError):
        pass
    return result
_shap_tree.decode_ubjson_buffer = _decode_ubjson_fixed

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_shap)

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, show=False)
plt.title(f'SHAP Summary — {best_label}')
plt.tight_layout()
plt.savefig(str(SHAP_DIR / f'shap_summary_{best_label}.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'SHAP summary salvo em shap_plots/')

In [ ]:
# Bar plot — importância média
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance — {best_label}')
plt.tight_layout()
plt.savefig(str(SHAP_DIR / f'shap_bar_{best_label}.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Salvando Modelo e Métricas

In [ ]:
save_model(best_model, '../model.pkl')
save_metrics({'best_label': best_label, 'all_horizons': all_metrics, 'thresholds': threshold_results}, '../metrics.json')
print('model.pkl e metrics.json salvos.')